In [1]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression, LassoCV, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error,  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.datasets import make_regression
from datetime import datetime

df = pd.read_csv(r"C:\Users\Negus\OneDrive\שולחן העבודה\customer_churn_dataset.csv")
df

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0
...,...,...,...,...,...,...,...,...,...,...,...,...
64369,64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64370,64371,37,Male,6,1,5,22,Standard,Annual,923,9,1
64371,64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1
64372,64373,50,Female,18,19,7,22,Standard,Monthly,540,13,1


In [2]:
#
df_copy = df.copy()
df_copy = df_copy.drop(columns = 'CustomerID')
df_copy

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,53,Female,58,24,9,2,Standard,Annual,533,18,0
...,...,...,...,...,...,...,...,...,...,...,...
64369,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64370,37,Male,6,1,5,22,Standard,Annual,923,9,1
64371,25,Male,39,14,8,30,Premium,Monthly,327,20,1
64372,50,Female,18,19,7,22,Standard,Monthly,540,13,1


In [5]:
#check for missing values in the dataset
df_copy.isnull().sum()

Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

In [7]:
#check for duplicate rows
is_duplicated = df_copy.duplicated().sum()
is_duplicated

0

In [9]:
df_copy = pd.get_dummies(df_copy, drop_first = True)
df_copy

,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_Male,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,22,25,14,4,27,598,9,1,False,False,False,True,False
1,41,28,28,7,13,584,20,0,False,False,True,True,False
2,47,27,10,2,29,757,21,0,True,True,False,False,False
3,35,9,12,5,17,232,18,0,True,True,False,False,True
4,53,58,24,9,2,533,18,0,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64369,45,33,12,6,21,947,14,1,False,False,False,False,True
64370,37,6,1,5,22,923,9,1,True,False,True,False,False
64371,25,39,14,8,30,327,20,1,True,True,False,True,False
64372,50,18,19,7,22,540,13,1,False,False,True,True,False


In [11]:
#correlations with the fetures and label
correlation_matrix = df_copy.corr()
correlation_matrix

,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_Male,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
Age,1.000000,-0.007763,-0.038331,0.005014,-0.016132,0.006490,-0.000148,0.063457,0.001800,-0.004582,0.006161,0.001311,-0.000585
Tenure,-0.007763,1.000000,0.023485,0.060065,0.055963,0.009474,0.005770,0.195327,0.029418,-0.002881,0.005078,-0.003306,0.002310
Usage Frequency,-0.038331,0.023485,1.000000,-0.014072,0.031132,0.001527,-0.009192,-0.115098,-0.006907,0.000364,-0.000744,0.008066,0.005677
Support Calls,0.005014,0.060065,-0.014072,1.000000,0.064298,0.021750,0.001666,0.304631,0.035418,-0.005009,-0.000250,-0.016492,0.005705
Payment Delay,-0.016132,0.055963,0.031132,0.064298,1.000000,-0.031119,-0.008076,0.557386,-0.058578,-0.003979,0.000680,0.028522,-0.012800
Total Spend,0.006490,0.009474,0.001527,0.021750,-0.031119,1.000000,-0.007692,-0.078867,0.029337,0.006925,-0.006608,0.024744,-0.006814
Last Interaction,-0.000148,0.005770,-0.009192,0.001666,-0.008076,-0.007692,1.000000,-0.002818,-0.000472,-0.005186,0.000662,0.000819,0.002925
Churn,0.063457,0.195327,-0.115098,0.304631,0.557386,-0.078867,-0.002818,1.000000,-0.164549,-0.012334,-0.000539,0.061464,-0.046000
Gender_Male,0.001800,0.029418,-0.006907,0.035418,-0.058578,0.029337,-0.000472,-0.164549,1.000000,0.000281,0.005380,-0.028741,0.006856
Subscription Type_Premium,-0.004582,-0.002881,0.000364,-0.005009,-0.003979,0.006925,-0.005186,-0.012334,0.000281,1.000000,-0.500122,0.003127,0.000304


In [13]:
# ⦁	Generate a pairplot that visualizes those correlations.

# sns.pairplot(df_copy, diag_kind = 'hist')

In [15]:
#feture corelations with the label
correlations_with_label = correlation_matrix['Churn']
correlations_with_label

Age                           0.063457
Tenure                        0.195327
Usage Frequency              -0.115098
Support Calls                 0.304631
Payment Delay                 0.557386
Total Spend                  -0.078867
Last Interaction             -0.002818
Churn                         1.000000
Gender_Male                  -0.164549
Subscription Type_Premium    -0.012334
Subscription Type_Standard   -0.000539
Contract Length_Monthly       0.061464
Contract Length_Quarterly    -0.046000
Name: Churn, dtype: float64

In [17]:
#feture corelations with each other 
correlations_ = correlation_matrix.drop("Churn", axis = 1)
correlations_

,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Gender_Male,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
Age,1.000000,-0.007763,-0.038331,0.005014,-0.016132,0.006490,-0.000148,0.001800,-0.004582,0.006161,0.001311,-0.000585
Tenure,-0.007763,1.000000,0.023485,0.060065,0.055963,0.009474,0.005770,0.029418,-0.002881,0.005078,-0.003306,0.002310
Usage Frequency,-0.038331,0.023485,1.000000,-0.014072,0.031132,0.001527,-0.009192,-0.006907,0.000364,-0.000744,0.008066,0.005677
Support Calls,0.005014,0.060065,-0.014072,1.000000,0.064298,0.021750,0.001666,0.035418,-0.005009,-0.000250,-0.016492,0.005705
Payment Delay,-0.016132,0.055963,0.031132,0.064298,1.000000,-0.031119,-0.008076,-0.058578,-0.003979,0.000680,0.028522,-0.012800
Total Spend,0.006490,0.009474,0.001527,0.021750,-0.031119,1.000000,-0.007692,0.029337,0.006925,-0.006608,0.024744,-0.006814
Last Interaction,-0.000148,0.005770,-0.009192,0.001666,-0.008076,-0.007692,1.000000,-0.000472,-0.005186,0.000662,0.000819,0.002925
Churn,0.063457,0.195327,-0.115098,0.304631,0.557386,-0.078867,-0.002818,-0.164549,-0.012334,-0.000539,0.061464,-0.046000
Gender_Male,0.001800,0.029418,-0.006907,0.035418,-0.058578,0.029337,-0.000472,1.000000,0.000281,0.005380,-0.028741,0.006856
Subscription Type_Premium,-0.004582,-0.002881,0.000364,-0.005009,-0.003979,0.006925,-0.005186,0.000281,1.000000,-0.500122,0.003127,0.000304


In [19]:
df_copy

,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_Male,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,22,25,14,4,27,598,9,1,False,False,False,True,False
1,41,28,28,7,13,584,20,0,False,False,True,True,False
2,47,27,10,2,29,757,21,0,True,True,False,False,False
3,35,9,12,5,17,232,18,0,True,True,False,False,True
4,53,58,24,9,2,533,18,0,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64369,45,33,12,6,21,947,14,1,False,False,False,False,True
64370,37,6,1,5,22,923,9,1,True,False,True,False,False
64371,25,39,14,8,30,327,20,1,True,True,False,True,False
64372,50,18,19,7,22,540,13,1,False,False,True,True,False


In [21]:
X = df_copy.drop('Churn', axis = 1)
Y = df_copy['Churn']

scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

#show the features in dataframe
scaled_X = pd.DataFrame(data = scaled_X, columns = X.columns)
scaled_X

X_train, X_test, y_train, y_test = train_test_split(scaled_X, Y, test_size=0.3, random_state=42)

log_model =  LogisticRegression()
log_model.fit(X_train, y_train)


y_pred = log_model.predict(X_test)

Accuracy = accuracy_score(y_test, y_pred)
Recall = recall_score(y_test, y_pred)
Precision = precision_score(y_test, y_pred)
F1_Score = 2 * (Precision * Recall) / (Precision + Recall)

print("Accuracy:", Accuracy)
print("-"*30)
print("Recall:", Recall)
print("-"*30)
print("F1 Score:", F1_Score)
print("-"*30)
print(classification_report(y_test, y_pred) )

Accuracy: 0.8290270802050432
------------------------------
Recall: 0.8260159058720994
------------------------------
F1 Score: 0.8211848803205893
------------------------------
              precision    recall  f1-score   support

           0       0.84      0.83      0.84     10134
           1       0.82      0.83      0.82      9179

    accuracy                           0.83     19313
   macro avg       0.83      0.83      0.83     19313
weighted avg       0.83      0.83      0.83     19313



In [23]:
# ⦁	Perform KNN with optimal K value.
cv_scores = []
for k in range(1,20):
    knn_model = KNeighborsClassifier(n_neighbors = k)
    score = cross_val_score(knn_model, X_train, y_train, cv = 15, scoring = 'accuracy')
    
    #append rhe mean scores of cv to cv_scores 
    cv_scores.append(score.mean())
    #+1 because the indexs start from 0
    optimal_k = cv_scores.index(max(cv_scores)) +1
    
knn_model = KNeighborsClassifier(n_neighbors = optimal_k)
knn_model.fit(X_train, y_train)

Knn_y_pred = knn_model.predict(X_test)
    
Knn_Accuracy = accuracy_score(y_test, Knn_y_pred)
Knn_Recall = recall_score(y_test, Knn_y_pred)
Knn_Precision = precision_score(y_test, Knn_y_pred)
Knn_F1_Score = 2 * (Knn_Precision * Knn_Recall) / (Knn_Precision + Knn_Recall)

print("Accuracy:", Knn_Accuracy)
print("-"*30)
print("Recall:", Knn_Recall)
print("-"*30)
print("F1 Score:", Knn_F1_Score)
print("-"*30)
print(classification_report(y_test, Knn_y_pred) )


Accuracy: 0.9131155180448403
------------------------------
Recall: 0.9260268003050441
------------------------------
F1 Score: 0.9101616875468466
------------------------------
              precision    recall  f1-score   support

           0       0.93      0.90      0.92     10134
           1       0.89      0.93      0.91      9179

    accuracy                           0.91     19313
   macro avg       0.91      0.91      0.91     19313
weighted avg       0.91      0.91      0.91     19313



In [25]:
# ⦁	Perform SVM with optimal parameters.
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Create an SVC model
svm = SVC()

# Define the parameter grid
param_grid = {'C': list(range(1, 5)),  # List of values for C
              'kernel': ['linear', 'rbf']}  # Correct spelling of 'kernel'

# Set up GridSearchCV to find the best parameters
grid = GridSearchCV(svm, param_grid)  # 5-fold cross-validation
grid.fit(X_train, y_train)

# Make predictions using the best found parameters
grid_pred = grid.predict(X_test)

print("Optimal params:", grid.best_estimator_ )
print("Accuracy:", accuracy_score(y_test, grid_pred))

Optimal params: SVC(C=4)
Accuracy: 0.9492569771656397


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score


rf_model = RandomForestClassifier(random_state=42)

# Step 2: Define the parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],           # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],           # Maximum depth of the tree
    'criterion': ['gini', 'entropy'],          # Function to measure the quality of a split
    'bootstrap': [True, False]                 # Whether bootstrap samples are used when building trees
}

#GridSearchCV to find the best parameters
rf_grid = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy')


rf_grid.fit(X_train, y_train)


rf_pred = rf_grid.predict(X_test)


rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_f1_score = f1_score(y_test, rf_pred)

# Output the best parameters and performance
print("Best Parameters:", rf_grid.best_estimator_)
print("Accuracy:", rf_accuracy)
print("Recall:", rf_recall)
print("F1 Score:", rf_f1_score)


Best Parameters: RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=20,
                       random_state=42)
Accuracy: 0.9990679852948791
Recall: 0.9982568907288376
F1 Score: 0.9990187527256869


In [33]:
model_errors = {
    'LOG': Accuracy,
    'KNN':  Knn_Accuracy,
    'SVM': accuracy_score(y_test, grid_pred),
    'Random': rf_accuracy
}
print(model_errors.values())
best_model = max(model_errors, key = model_errors.get )
print(f"Best model based on Accuracy: {best_model}")

dict_values([0.8290270802050432, 0.9131155180448403, 0.9492569771656397, 0.9990679852948791])
Best model based on Accuracy: Random


In [43]:
from joblib import dump, load
final_model = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy')
final_model.fit(X, Y)

dump(final_model, 'final_model.joblib')
dump(scaler, 'scaler.joblib')

loaded_model = load('final_model.joblib')
loaded_scaler = load('scaler.joblib')


new_data = pd.DataFrame({
    'Age': [35, 45, 50], 
    'Tenure': [5, 10, 3],
    'Usage Frequency': [20, 15, 25],
    'Support Calls': [2, 1, 3],
    'Payment Delay': [5, 0, 2],
    'Total Spend': [1000, 500, 1200],
    'Last Interaction': [30, 15, 60],
    'Gender_Male': [1, 0, 1],
    'Subscription Type_Premium': [1, 0, 0],
    'Subscription Type_Standard': [0, 0, 1],  
    'Contract Length_Monthly': [0, 1, 0],
    'Contract Length_Quarterly': [0, 0, 1],  

})
# Scale the new data using the loaded scaler
new_data_scaled = loaded_scaler.transform(new_data)

# Make predictions using the trained model
predictions = loaded_model.predict(new_data_scaled)
print(new_data)
print('---------------------')
# Output the predictions
print("New Data Predictions:", predictions)

   Age  Tenure  Usage Frequency  Support Calls  Payment Delay  Total Spend  \
0   35       5               20              2              5         1000   
1   45      10               15              1              0          500   
2   50       3               25              3              2         1200   

   Last Interaction  Gender_Male  Subscription Type_Premium  \
0                30            1                          1   
1                15            0                          0   
2                60            1                          0   

   Subscription Type_Standard  Contract Length_Monthly  \
0                           0                        0   
1                           0                        1   
2                           1                        0   

   Contract Length_Quarterly  
0                          0  
1                          0  
2                          1  
---------------------
New Data Predictions: [1 1 0]


C:\Users\Negus\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
